In [ ]:
    ############    #############   Routers, services, repositories   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.2 Backend Engineering
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Routers, Services, Repositories   #############   ##############   

 =>  This is the layered architecture from the diagram below, applied concretely in
       FastAPI: a router handles HTTP concerns only, a service holds business logic, a
       repository handles data access only.

 =>  The payoff: you can unit-test the service layer with a fake repository (no real
       database needed), and swap the repository's storage (Postgres, in-memory, a mock)
       without touching business logic at all.


<img src="images/layered-architecture.png" alt="Layered backend architecture: router, service, repository, database">

In [ ]:
from fastapi import FastAPI, APIRouter, Depends
from fastapi.testclient import TestClient
from pydantic import BaseModel

class User(BaseModel):
    id: int
    email: str

# ---- Repository: data access only ----
class UserRepository:
    def __init__(self):
        self._users = {1: User(id=1, email="a@b.com")}

    def get(self, user_id: int) -> User | None:
        return self._users.get(user_id)

# ---- Service: business logic ----
class UserService:
    def __init__(self, repo: UserRepository):
        self._repo = repo

    def get_user_or_raise(self, user_id: int) -> User:
        user = self._repo.get(user_id)
        if user is None:
            raise ValueError(f"user {user_id} not found")
        return user

# ---- Router: HTTP concerns only ----
def get_user_service() -> UserService:
    return UserService(UserRepository())

router = APIRouter()

@router.get("/users/{user_id}", response_model=User)
def get_user(user_id: int, service: UserService = Depends(get_user_service)) -> User:
    return service.get_user_or_raise(user_id)

app = FastAPI()
app.include_router(router)
# raise_server_exceptions=False: without this, TestClient re-raises the underlying
# exception into the TEST itself (great for debugging in pytest); with it, you get back
# the actual HTTP response a real client would see -- a bare 500.
client = TestClient(app, raise_server_exceptions=False)

print(client.get("/users/1").json())
print(client.get("/users/999").status_code)  # ValueError isn't handled yet -> 500


In [ ]:
 =>  Notice the last call returns 500, not a clean 404 -- the router doesn't yet catch
       the service's ValueError. That's intentional here: see the Exception Architecture
       notebook (Phase 0.1) for the proper fix (a domain exception + exception handler).

 =>  By default, FastAPI's TestClient RE-RAISES an unhandled exception into your test
       process instead of turning it into a 500 response -- that's deliberately useful for
       debugging (you get the real traceback in pytest). Pass
       'raise_server_exceptions=False' when you specifically want to assert on the HTTP
       response a real client would receive, as done above.

 =>  In a real app, get_user_service would build UserService with a real database-backed
       repository (e.g. one wrapping SQLAlchemy) -- the router and service code above
       wouldn't change at all.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Write a unit test for UserService using a FakeUserRepository (a plain dict-backed
           stand-in), with NO FastAPI or HTTP involved at all.

 =>  [ ] Wire the Exception Architecture pattern (Phase 0.1) into this router so a missing
           user returns a clean 404 instead of a 500.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Putting business logic (validation rules, calculations) directly in the router
       function -- this makes it untestable without spinning up FastAPI/HTTP.

 =>  Letting the repository leak framework-specific types (a FastAPI Request, an ORM
       session) up into the service layer -- the service should only depend on plain
       Python/domain types.
